# NB12 — Revision analyses requested in peer review

**Purpose.** Runs every patient-level analysis needed to answer the internal review, using only files already on Drive (no BigQuery). (A) chance ceiling for the 0.8-1.2 band and for funnel plots under perfect calibration; (B) unstratified random-split sensitivity (the earlier split was stratified on outcome); (C) empirical-Bayes variants: leave-hospital-out prior, no global step, and a one-stage random-intercept GLMM; bootstrap CIs for tau and Holm-adjusted tests; (D) leave-region-out transport of the network prior; (E) label budget across all eligible hospitals; (F) meta-regression with proportion of tau^2 explained; (G) why the MIMIC temporal AUROC exceeds the development AUROC; (H) second outcome: ICU stay >= 7 days.

**Research question.** Peer-review revision

| | |
|---|---|
| **Inputs** | `models/pred_eicu.parquet`, `models/pred_eval_strategies.parquet`, `models/pred_mimic.parquet`, analysis parquet, raw cohort parquet (for ICU length of stay) |
| **Outputs** | `results/NB12/*.csv`, figures, `results/NB12_results.zip` |
| **Free-Colab runtime** | 30-60 min (checkpointed) |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB12')

### 0. Load everything

In [ ]:
import json, joblib, lightgbm as lgb
from scipy import stats, optimize
from scipy.stats import wilcoxon, spearmanr
import statsmodels.api as sm
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
evs0 = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies.parquet'))
pm = pd.read_parquet(os.path.join(tu.MODELS, 'pred_mimic.parquet'))
bp = json.load(open(os.path.join(tu.MODELS, 'lgbm_best_params.json')))
FEATS = tu.feature_set('V1')
hinfo = e.groupby('hospital_id').agg(numbedscategory=('numbedscategory', 'first'),
                                     teachingstatus=('teachingstatus', 'first'), region=('region', 'first'),
                                     casemix_apache=('apache_pred', 'mean'), mean_age=('age', 'mean')).reset_index()
hy = pd.read_csv(os.path.join(tu.RESULTS, 'NB03', 'hospital_year_counts.csv'))
ELIG = sorted(set(hy.loc[hy['eligible_temporal'], 'hospital_id']))
upd = pred[pred['split'] == 'update'].reset_index(drop=True)
ev = pred[pred['split'] == 'eval'].reset_index(drop=True)
B = tu.n_boot(1000)
R_SIM = 100 if tu.FAST else 1000
R_LB = 3 if tu.FAST else 50
L = dict(tu.STRATEGY_LABELS)
L.update({'S3_eb_loho': 'EB, leave-hospital-out prior', 'S3_eb_noglobal_loho': 'EB without global step (LOHO prior)',
          'S3_glmm': 'One-stage random-intercept GLMM', 'S3_glmm_loho': 'One-stage GLMM, leave-hospital-out'})
print('eligible hospitals (temporal):', len(ELIG), '| update', len(upd), '| eval', len(ev))

### Shared helpers (per-hospital summaries, chance ceiling, EB / GLMM fitting)

In [ ]:
def hosp_summary(df, pcol, label, hosp, ycol='mortality'):
    ph = tu.per_hospital(df[df['hospital_id'].isin(hosp)], pcol, ycol=ycol, min_n=0, min_events=0, full=False)
    ma = tu.meta_analysis(ph['ln_oe'], ph['ln_oe_se'])
    return ph, dict(strategy=label, hospitals=len(ph), pct_acceptable=100 * ph['oe'].between(*tu.OE_ACCEPT).mean(),
                    median_abs_ln_oe=ph['ln_oe'].abs().median(), tau=ma['tau'], pooled_oe_re=np.exp(ma['mu']),
                    pi_lo=np.exp(ma['pi_lo']), pi_hi=np.exp(ma['pi_hi']), i2=ma['i2'])

def chance_ceiling(df, pcol, hosp, R=R_SIM, seed=7):
    """Simulate each hospital's deaths under PERFECT calibration (y ~ Bernoulli(p)) and recompute every
    hospital-level statistic. Gives the best value any updating strategy could reach by chance alone."""
    rng = np.random.default_rng(seed)
    g = {h: d[pcol].values for h, d in df[df['hospital_id'].isin(hosp)].groupby('hospital_id')}
    H = list(g)
    O = np.zeros((R, len(H))); E = np.array([g[h].sum() for h in H]); N = np.array([len(g[h]) for h in H])
    for j, h in enumerate(H):
        p = g[h]
        for r0 in range(0, R, 250):                       # chunks keep memory small
            r1 = min(R, r0 + 250)
            O[r0:r1, j] = (rng.random((r1 - r0, len(p))) < p).sum(1)
    oe = O / E
    inband = 100 * ((oe >= tu.OE_ACCEPT[0]) & (oe <= tu.OE_ACCEPT[1])).mean(1)
    med = np.median(np.abs(np.log(np.maximum(O, 0.5) / E)), axis=1)
    lo95, hi95 = stats.poisson.ppf(0.025, E), stats.poisson.ppf(0.975, E)
    lo998, hi998 = stats.poisson.ppf(0.001, E), stats.poisson.ppf(0.999, E)
    out95 = 100 * ((O < lo95) | (O > hi95)).mean(1)
    out998 = 100 * ((O < lo998) | (O > hi998)).mean(1)
    taus = []
    for r in range(min(R, 200)):
        o = np.maximum(O[r], 0.5)
        taus.append(tu.meta_analysis(np.log(o / E), np.sqrt(np.clip(1 - o / N, 1e-9, None) / o), hksj=False)['tau'])
    taus = np.array(taus)
    return dict(hospitals=len(H), ceiling_pct_acceptable=inband.mean(), ceiling_lo=np.percentile(inband, 2.5),
                ceiling_hi=np.percentile(inband, 97.5), null_median_abs_ln_oe=med.mean(),
                null_pct_outside_95=out95.mean(), null_pct_outside_998=out998.mean(),
                null_tau_mean=taus.mean(), null_tau_pct_zero=100 * (taus == 0).mean())

def by_hosp(df, pcol, ycol='mortality'):
    return {h: (d[pcol].values, d[ycol].values.astype(float)) for h, d in df.groupby('hospital_id')}

def network_prior(groups, exclude=None, global_step=True):
    """Global logistic recalibration + DerSimonian-Laird (mu, tau^2) of site intercepts, from all sites
    in `groups` except `exclude` (a hospital id, or a set of ids)."""
    ex = exclude if isinstance(exclude, (set, frozenset)) else ({exclude} if exclude is not None else set())
    keys = [h for h in groups if h not in ex]
    if global_step:
        p = np.concatenate([groups[h][0] for h in keys]); y = np.concatenate([groups[h][1] for h in keys])
        ag, bg = tu.fit_logistic(p, y)
    else:
        ag, bg = 0.0, 1.0
    t = np.array([tu.fit_intercept(tu.apply_logistic(groups[h][0], ag, bg), groups[h][1]) for h in keys])
    ma = tu.meta_analysis(t[:, 0], t[:, 1], hksj=False)
    return dict(ag=ag, bg=bg, mu=ma['mu'], tau2=ma['tau2'])

def eb_apply(prior, p_upd, y_upd, p_new):
    a, se = tu.fit_intercept(tu.apply_logistic(p_upd, prior['ag'], prior['bg']), y_upd)
    a_eb = tu.eb_shrink_single(a, se, prior['mu'], prior['tau2'])
    return tu.expit(tu.logit(tu.apply_logistic(p_new, prior['ag'], prior['bg'])) + a_eb), a, se, a_eb

GH_X, GH_W = np.polynomial.hermite.hermgauss(15)
class RIGLMM:
    """One-stage random-intercept logistic recalibration
         logit P(y=1) = alpha + beta * logit(p) + u_h,   u_h ~ N(0, tau^2),
    fitted by maximum likelihood with adaptive Gauss-Hermite quadrature (15 nodes)."""
    def __init__(self, p, y, h):
        o = np.argsort(np.asarray(h), kind='stable')
        self.lp = tu.logit(np.asarray(p, float)[o]); self.y = np.asarray(y, float)[o]; hh = np.asarray(h)[o]
        self.sites, self.start = np.unique(hh, return_index=True)
        self.idx = np.repeat(np.arange(len(self.sites)), np.diff(np.append(self.start, len(hh))))
    def _mode(self, eta, tau2):
        u = np.zeros(len(self.sites))
        for _ in range(50):
            mu = tu.expit(eta + u[self.idx])
            g = np.add.reduceat(self.y - mu, self.start) - u / tau2
            H = np.add.reduceat(mu * (1 - mu), self.start) + 1 / tau2
            st = g / H; u = u + st
            if np.max(np.abs(st)) < 1e-9:
                break
        mu = tu.expit(eta + u[self.idx])
        return u, 1 / np.sqrt(np.add.reduceat(mu * (1 - mu), self.start) + 1 / tau2)
    def _nodes(self, theta):
        a, b, lt = theta; tau2 = np.exp(2 * lt); eta = a + b * self.lp
        u, s = self._mode(eta, tau2)
        Ls, Us = [], []
        for x in GH_X:
            uk = u + np.sqrt(2) * s * x
            z = eta + uk[self.idx]
            ll = np.add.reduceat(self.y * z - np.logaddexp(0, z), self.start)
            Ls.append(ll - uk ** 2 / (2 * tau2) - 0.5 * np.log(2 * np.pi * tau2) + x ** 2); Us.append(uk)
        return np.log(GH_W)[:, None] + np.array(Ls), np.array(Us), s
    def negll(self, theta):
        lw, _, s = self._nodes(theta)
        return -np.sum(np.logaddexp.reduce(lw, axis=0) + np.log(np.sqrt(2) * s))
    def fit(self, x0=(0.0, 1.0, np.log(0.3))):
        r = optimize.minimize(self.negll, np.array(x0, float), method='L-BFGS-B',
                              bounds=[(-5, 5), (0.2, 3.0), (np.log(1e-3), np.log(3.0))])
        self.theta, self.ok = r.x, bool(r.success)
        self.alpha, self.beta, self.tau = r.x[0], r.x[1], float(np.exp(r.x[2]))
        lw, U, _ = self._nodes(r.x)
        wn = np.exp(lw - np.logaddexp.reduce(lw, axis=0))
        self.u_post = dict(zip(self.sites, np.sum(wn * U, axis=0)))
        return self
    def posterior_u(self, p, y):
        """Posterior mean of a NEW site's intercept given its own data and the fitted (alpha, beta, tau)."""
        one = RIGLMM(p, y, np.zeros(len(p)))
        lw, U, _ = one._nodes(self.theta)
        wn = np.exp(lw - np.logaddexp.reduce(lw, axis=0))
        return float(np.sum(wn * U, axis=0)[0])
    def predict(self, p, u):
        return tu.expit(self.alpha + self.beta * tu.logit(p) + u)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty(len(p)); run = 0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (len(p) - i) * p[j])); adj[j] = run
    return adj
print('helpers ready')

## A. Chance ceiling: what would a perfectly calibrated model show?

In [ ]:
ph_all = tu.per_hospital(pred, 'p_lgbm')                        # all eICU years, as in NB06
H137 = ph_all.loc[ph_all['eligible'], 'hospital_id'].tolist()
H194 = ph_all.loc[ph_all['expected'] >= 1, 'hospital_id'].tolist()
rows = []
c = tu.cached('ceiling_137', chance_ceiling, pred, 'p_lgbm', H137)
rows.append(dict(analysis='Heterogeneity set (2014+2015, k=137)', strategy='No updating', **c,
                 observed_pct_acceptable=100 * ph_all.loc[ph_all['eligible'], 'oe'].between(*tu.OE_ACCEPT).mean(),
                 observed_median_abs_ln_oe=ph_all.loc[ph_all['eligible'], 'ln_oe'].abs().median()))
c = tu.cached('ceiling_194', chance_ceiling, pred, 'p_lgbm', H194)
v = ph_all[ph_all['expected'] >= 1]
rows.append(dict(analysis='Funnel set (expected deaths >= 1, k=194)', strategy='No updating', **c,
                 observed_pct_outside_95=100 * tu.outside_funnel(v, 0.95).mean(),
                 observed_pct_outside_998=100 * tu.outside_funnel(v, 0.998).mean()))
STRATS = [s for s in ['S0_none', 'S1a_global_intercept', 'S1b_global_logistic', 'S1c_global_isotonic',
                      'S2a_local_intercept', 'S2b_local_logistic', 'S2c_local_isotonic', 'S3_eb', 'S4_apache',
                      'S5_refit'] if s in evs0]
for s in STRATS:
    d = evs0 if s != 'S4_apache' else evs0[evs0['S4_apache'].notna()]
    c = tu.cached(f'ceiling_97_{s}', chance_ceiling, d, s, ELIG)
    ph, sm_ = hosp_summary(d, s, L[s], ELIG)
    rows.append(dict(analysis='Temporal set (2015, k=97)', strategy=L[s], **c,
                     observed_pct_acceptable=sm_['pct_acceptable'], observed_median_abs_ln_oe=sm_['median_abs_ln_oe']))
ceil = pd.DataFrame(rows)
tu.save_table(ceil.round(3), 'A_chance_ceiling')

## B. Random within-hospital split, NOT stratified on outcome\nThe earlier sensitivity split (NB03) was stratified by outcome within each hospital, so the update half had exactly half of each hospital's deaths. An intercept fitted on one half then reproduces the other half's death rate almost by construction. Here the split is a plain random 50/50 within hospital.

In [ ]:
KEY = ['S0_none', 'S1b_global_logistic', 'S2a_local_intercept', 'S2b_local_logistic', 'S3_eb']
def strategy_summary(upd_, ev_, pcol, label, min_n=tu.MIN_HOSP_N, min_events=tu.MIN_HOSP_EVENTS, ycol='mortality',
                     keys=KEY, ceiling=True):
    evs_, info, _ = tu.run_strategies(upd_, ev_, pcol, ycol=ycol, apache_col='__none__')
    c_ = evs_.groupby('hospital_id').agg(n=(ycol, 'size'), d=(ycol, 'sum'))
    nu = upd_.groupby('hospital_id').size()
    elig = [h for h in c_[(c_.n >= min_n) & (c_.d >= min_events)].index if nu.get(h, 0) >= tu.MIN_UPDATE_N]
    out = []
    for s in keys:
        _, r = hosp_summary(evs_, s, L[s], elig, ycol=ycol)
        r['analysis'] = label
        if ceiling:
            cc = chance_ceiling(evs_.rename(columns={ycol: '_y'}), s, elig, R=min(R_SIM, 500))
            r['ceiling_pct_acceptable'] = cc['ceiling_pct_acceptable']
        out.append(r)
    return out, evs_, elig, info

rng = np.random.default_rng(tu.SEED)
pred['random_split_unstrat'] = 'eval'
for h, idx in pred.groupby('hospital_id').groups.items():
    idx = np.array(list(idx)); rng.shuffle(idx)
    pred.loc[idx[: len(idx) // 2], 'random_split_unstrat'] = 'update'
rs = []
for col, lab in [('random_split', 'Random 50/50 split, stratified on outcome (as in first draft)'),
                 ('random_split_unstrat', 'Random 50/50 split, unstratified (corrected)')]:
    o, _, el, _ = strategy_summary(pred[pred[col] == 'update'].reset_index(drop=True),
                                   pred[pred[col] == 'eval'].reset_index(drop=True), 'p_lgbm', lab)
    rs += o
    print(lab, '| hospitals', len(el))
rs = pd.DataFrame(rs)
tu.save_table(rs.round(3), 'B_random_split')

## C. Empirical-Bayes variants in the primary 2014 -> 2015 comparison\n* **EB (primary, as in NB08):** network quantities from all update hospitals (standard empirical Bayes; the target hospital's own 2014 data enter its prior, but no 2015 data are used).\n* **EB, leave-hospital-out (LOHO):** global recalibration, mu and tau^2 from all *other* hospitals.\n* **EB without global step:** shrink intercepts of the unadjusted model (no global logistic recalibration).\n* **One-stage GLMM:** random-intercept logistic recalibration fitted jointly (with and without LOHO).

In [ ]:
G = by_hosp(upd, 'p_lgbm')
E_ = by_hosp(ev, 'p_lgbm')
def loho_priors():
    out = {}
    for h in ELIG:
        out[(h, True)] = network_prior(G, exclude=h, global_step=True)
        out[(h, False)] = network_prior(G, exclude=h, global_step=False)
    return out
PRI = tu.cached('loho_priors', loho_priors)
evs = evs0.copy()
for col in ['S3_eb_loho', 'S3_eb_noglobal_loho']:
    evs[col] = np.nan
prior_rows = []
for h in ELIG:
    msk = (evs['hospital_id'] == h).values
    for col, gs in [('S3_eb_loho', True), ('S3_eb_noglobal_loho', False)]:
        pr = PRI[(h, gs)]
        pp, a, se, a_eb = eb_apply(pr, G[h][0], G[h][1], evs.loc[msk, 'p_lgbm'].values)
        evs.loc[msk, col] = pp
        prior_rows.append(dict(hospital_id=h, variant=col, mu=pr['mu'], tau=np.sqrt(pr['tau2']), a_local=a,
                               se_local=se, a_shrunk=a_eb, shrink_B=pr['tau2'] / (pr['tau2'] + se ** 2)))
tu.save_table(pd.DataFrame(prior_rows).round(4), 'C_eb_loho_priors', show=False)

# one-stage GLMM, all update hospitals
def fit_glmm_all():
    return RIGLMM(upd['p_lgbm'].values, upd['mortality'].values, upd['hospital_id'].values).fit()
gl = tu.cached('glmm_all', fit_glmm_all)
print(f'GLMM (all hospitals): alpha={gl.alpha:.3f} beta={gl.beta:.3f} tau={gl.tau:.3f} converged={gl.ok}')
evs['S3_glmm'] = gl.predict(evs['p_lgbm'].values, np.array([gl.u_post.get(h, 0.0) for h in evs['hospital_id']]))
def fit_glmm_loho():
    res = {}
    for i, h in enumerate(ELIG):
        o = upd[upd['hospital_id'] != h]
        g_ = RIGLMM(o['p_lgbm'].values, o['mortality'].values, o['hospital_id'].values).fit(x0=gl.theta)
        res[h] = dict(theta=g_.theta, u=g_.posterior_u(G[h][0], G[h][1]), ok=g_.ok)
        if (i + 1) % 20 == 0:
            print(f'  GLMM LOHO {i+1}/{len(ELIG)}', flush=True)
    return res
GLO = tu.cached('glmm_loho', fit_glmm_loho)
evs['S3_glmm_loho'] = np.nan
for h in ELIG:
    msk = (evs['hospital_id'] == h).values
    a_, b_, lt_ = GLO[h]['theta']
    evs.loc[msk, 'S3_glmm_loho'] = tu.expit(a_ + b_ * tu.logit(evs.loc[msk, 'p_lgbm'].values) + GLO[h]['u'])
tu.save_table(pd.DataFrame([dict(model='GLMM all hospitals', alpha=gl.alpha, beta=gl.beta, tau=gl.tau, converged=gl.ok),
                            dict(model='GLMM LOHO (median over hospitals)', alpha=np.median([v['theta'][0] for v in GLO.values()]),
                                 beta=np.median([v['theta'][1] for v in GLO.values()]),
                                 tau=np.median([np.exp(v['theta'][2]) for v in GLO.values()]),
                                 converged=all(v['ok'] for v in GLO.values()))]).round(4), 'C_glmm_params')
ALL = STRATS + ['S3_eb_loho', 'S3_eb_noglobal_loho', 'S3_glmm', 'S3_glmm_loho']
evs.to_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies_rev.parquet'), index=False)

In [ ]:
# Summary, chance ceiling, bootstrap CI for tau (hospital resampling, conditional on fitted updates)
phs, rows = {}, []
for s in ALL:
    d = evs if s != 'S4_apache' else evs[evs['S4_apache'].notna()]
    phs[s], r = hosp_summary(d, s, L[s], ELIG)
    if s not in STRATS:
        r.update({k: v for k, v in tu.cached(f'ceiling_97_{s}', chance_ceiling, d, s, ELIG).items()
                  if k in ('ceiling_pct_acceptable', 'ceiling_lo', 'ceiling_hi', 'null_median_abs_ln_oe')})
    else:
        r.update({k: v for k, v in tu.cached(f'ceiling_97_{s}', chance_ceiling, d, s, ELIG).items()
                  if k in ('ceiling_pct_acceptable', 'ceiling_lo', 'ceiling_hi', 'null_median_abs_ln_oe')})
    r['key'] = s
    rows.append(r)
est = pd.DataFrame({s: phs[s].set_index('hospital_id')['ln_oe'] for s in ALL})
se_ = pd.DataFrame({s: phs[s].set_index('hospital_id')['ln_oe_se'] for s in ALL})
rng = np.random.default_rng(42); H = est.index.values; bt = {s: [] for s in ALL}
for b in range(B):
    idx = rng.choice(H, len(H), replace=True)
    for s in ALL:
        bt[s].append(tu.meta_analysis(est.loc[idx, s], se_.loc[idx, s], hksj=False)['tau'])
for r in rows:
    bb = np.array(bt[r['key']]); r['tau_lo'], r['tau_hi'] = np.nanpercentile(bb, 2.5), np.nanpercentile(bb, 97.5)
summ = pd.DataFrame(rows)
tu.save_table(summ.round(4), 'C_strategies_extended')
diff = []
for ref in ['S3_eb', 'S3_eb_loho']:
    for s in [x for x in ALL if x not in (ref, 'S4_apache')]:
        t0 = summ.set_index('key').loc[s, 'tau'] - summ.set_index('key').loc[ref, 'tau']
        db = np.array(bt[s]) - np.array(bt[ref])
        diff.append(dict(reference=L[ref], other=L[s], tau_other_minus_ref=t0, lo=np.nanpercentile(db, 2.5),
                         hi=np.nanpercentile(db, 97.5), p_boot=2 * min((db <= 0).mean(), (db >= 0).mean())))
tu.save_table(pd.DataFrame(diff).round(4), 'C_tau_paired_differences')
# Wilcoxon on |ln O/E|: EB vs every other strategy, Holm-adjusted within the family
a = est.abs(); wx = []
for s in [x for x in ALL if x != 'S3_eb']:
    d = a[['S3_eb', s]].dropna()
    wx.append(dict(comparison=f'EB vs {L[s]}', k=len(d), median_eb=d['S3_eb'].median(), median_other=d[s].median(),
                   pct_hospitals_eb_better=100 * (d['S3_eb'] < d[s]).mean(), p=wilcoxon(d['S3_eb'], d[s]).pvalue))
wx = pd.DataFrame(wx); wx['p_holm'] = holm(wx['p'].values)
tu.save_table(wx.round(5), 'C_wilcoxon_holm')

In [ ]:
# Pooled performance of the new variants (hospital-cluster bootstrap), for the pooled table
pool = []
for s in ['S3_eb_loho', 'S3_eb_noglobal_loho', 'S3_glmm', 'S3_glmm_loho']:
    ci, _ = tu.bootstrap(evs['mortality'].values, evs[s].fillna(evs['S3_eb']).values, clusters=evs['hospital_id'].values,
                         B=tu.n_boot(500), metrics=['auroc', 'brier', 'oe', 'citl', 'slope', 'ici'])
    for _, r in ci.iterrows():
        pool.append(dict(strategy=L[s], metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi'],
                         note='non-eligible hospitals use primary EB' if 'loho' in s else ''))
tu.save_table(pd.DataFrame(pool).round(4), 'C_pooled_new_variants')
# Within-hospital discrimination is unchanged by any intercept-only hospital shift (monotone within hospital):
wa = []
for s in ['S0_none', 'S1b_global_logistic', 'S3_eb']:
    wa.append(dict(strategy=L[s], median_within_hospital_auroc=phs[s]['auroc'].median()))
tu.save_table(pd.DataFrame(wa).round(4), 'C_within_hospital_auroc')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
order = ['S0_none', 'S1b_global_logistic', 'S2a_local_intercept', 'S2b_local_logistic', 'S3_eb', 'S3_eb_loho',
         'S3_eb_noglobal_loho', 'S3_glmm', 'S5_refit', 'S4_apache']
order = [s for s in order if s in phs]
data = [phs[s]['oe'].dropna().clip(0.2, 5).values for s in order]
ax.axhspan(*tu.OE_ACCEPT, color=tu.C['aqua'], alpha=0.1, label='0.8-1.2 band')
bp_ = ax.boxplot(data, widths=0.5, showfliers=False, patch_artist=True, medianprops=dict(color=tu.C['ink']))
for patch in bp_['boxes']:
    patch.set_facecolor('white'); patch.set_edgecolor(tu.C['ink2'])
rr = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(i + 1 + rr.uniform(-0.18, 0.18, len(d)), d, s=6, color=tu.C['blue'], alpha=0.5, zorder=3)
sk = summ.set_index('key')
labels = [f"{L[s]}\n{sk.loc[s,'pct_acceptable']:.0f}% in band (ceiling {sk.loc[s,'ceiling_pct_acceptable']:.0f}%)" for s in order]
ax.set_yscale('log'); ax.axhline(1, color=tu.C['ink2'], lw=1)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=7)
ax.set_ylabel('Hospital O/E in 2015 (log scale)'); ax.legend(loc='lower left')
ax.set_title(f'Per-hospital calibration after each updating strategy (k={len(ELIG)})')
tu.save_fig(fig, 'fig5_strategies_per_hospital_rev'); plt.show()

## D. Leave-region-out: does the network prior transport to a new region?

In [ ]:
reg = hinfo.set_index('hospital_id')['region'].fillna('Unknown')
regions = [r for r in ['Midwest', 'Northeast', 'South', 'West'] if (reg.reindex(ELIG) == r).sum() > 0]
lro = []
def run_lro():
    out = []
    for R_ in regions:
        held = [h for h in ELIG if reg.get(h) == R_]
        excl = set(reg[reg == R_].index)
        for gs in (True, False):
            pr = network_prior(G, exclude=frozenset(excl), global_step=gs)
            for h in held:
                pe = E_[h][0]
                pp, a, se, a_eb = eb_apply(pr, G[h][0], G[h][1], pe)
                out.append(dict(region=R_, hospital_id=h, variant='EB' if gs else 'EB without global step',
                                p_pred=pp, y=E_[h][1], p_glob=tu.apply_logistic(pe, pr['ag'], pr['bg']) if gs else None,
                                tau_prior=np.sqrt(pr['tau2'])))
    return out
LRO = tu.cached('leave_region_out', run_lro)
def summ_from(ps, label):
    ph = []
    for h, y, p in ps:
        o, ex = y.sum(), p.sum()
        ph.append(dict(hospital_id=h, oe=o / ex, ln_oe=np.log(o / ex), ln_oe_se=np.sqrt((1 - o / len(y)) / o)))
    ph = pd.DataFrame(ph); ma = tu.meta_analysis(ph['ln_oe'], ph['ln_oe_se'])
    return dict(strategy=label, hospitals=len(ph), pct_acceptable=100 * ph['oe'].between(*tu.OE_ACCEPT).mean(),
                median_abs_ln_oe=ph['ln_oe'].abs().median(), tau=ma['tau'], pooled_oe_re=np.exp(ma['mu']),
                pi_lo=np.exp(ma['pi_lo']), pi_hi=np.exp(ma['pi_hi']))
heldH = sorted({r['hospital_id'] for r in LRO})
rows = [summ_from([(h, E_[h][1], E_[h][0]) for h in heldH], 'No updating')]
rows.append(summ_from([(r['hospital_id'], r['y'], r['p_glob']) for r in LRO if r['variant'] == 'EB'],
                      'Global logistic fitted on other regions'))
for vname in ['EB', 'EB without global step']:
    rows.append(summ_from([(r['hospital_id'], r['y'], r['p_pred']) for r in LRO if r['variant'] == vname],
                          f'{vname}, prior from other regions'))
for s in ['S2a_local_intercept', 'S3_eb', 'S3_eb_noglobal_loho']:
    g_ = evs[evs['hospital_id'].isin(heldH)]
    rows.append(summ_from([(h, d['mortality'].values, d[s].values) for h, d in g_.groupby('hospital_id')],
                          L[s] + ' (whole network)'))
lro_tab = pd.DataFrame(rows)
tu.save_table(lro_tab.round(4), 'D_leave_region_out')
print('held-out hospitals by region:', {R_: sum(reg.get(h) == R_ for h in heldH) for R_ in regions})

## E. Label budget across ALL eligible hospitals (n capped at each hospital's available update data)

In [ ]:
NS = [25, 50, 100, 200, 400]
nupd = {h: len(G[h][1]) for h in ELIG}
def lb_repeat(r):
    rng_ = np.random.default_rng(5000 + r); rows_ = []
    for h in ELIG:
        pu, yu = G[h]; pe, ye = E_[h]
        for n in NS:
            if nupd[h] < n:
                continue
            ii = rng_.choice(len(yu), n, replace=False); ps, ys = pu[ii], yu[ii]
            preds = {}
            a, _ = tu.fit_intercept(ps, ys); preds['Local intercept'] = tu.apply_logistic(pe, a)
            aa, bb = tu.fit_logistic(ps, ys)
            if not (np.isfinite(aa) and np.isfinite(bb) and bb > 0):
                aa, bb = a, 1.0
            preds['Local logistic'] = tu.apply_logistic(pe, aa, bb)
            preds['EB (network prior, LOHO)'] = eb_apply(PRI[(h, True)], ps, ys, pe)[0]
            preds['EB without global step (LOHO)'] = eb_apply(PRI[(h, False)], ps, ys, pe)[0]
            for k, pp in preds.items():
                o, ex = ye.sum(), pp.sum()
                rows_.append(dict(rep=r, n=n, method=k, hospital_id=h, oe=o / ex, abs_ln_oe=abs(np.log(o / ex))))
    return pd.DataFrame(rows_)
lb = pd.concat([tu.cached(f'lb_rep_{r:03d}', lb_repeat, r) for r in range(R_LB)], ignore_index=True)
base = []
for h in ELIG:
    pe, ye = E_[h]; o = ye.sum()
    for k, pp in [('No updating', pe), ('Global logistic (LOHO)', tu.apply_logistic(pe, PRI[(h, True)]['ag'], PRI[(h, True)]['bg']))]:
        base.append(dict(hospital_id=h, method=k, oe=o / pp.sum(), abs_ln_oe=abs(np.log(o / pp.sum()))))
base = pd.DataFrame(base)
FIX35 = [h for h in ELIG if nupd[h] >= 400]
out = []
for setname, hs_fn in [('all hospitals with >= n update patients', lambda n: [h for h in ELIG if nupd[h] >= n]),
                       (f'fixed set (>= 400 update patients, k={len(FIX35)})', lambda n: FIX35)]:
    for n in NS:
        hs = hs_fn(n)
        d = lb[(lb['n'] == n) & lb['hospital_id'].isin(hs)]
        for k, g_ in d.groupby('method'):
            out.append(dict(hospital_set=setname, n=n, hospitals=len(hs), method=k,
                            pct_acceptable=100 * g_['oe'].between(*tu.OE_ACCEPT).mean(),
                            median_abs_ln_oe=g_['abs_ln_oe'].median()))
        bb = base[base['hospital_id'].isin(hs)]
        for k, g_ in bb.groupby('method'):
            out.append(dict(hospital_set=setname, n=n, hospitals=len(hs), method=k,
                            pct_acceptable=100 * g_['oe'].between(*tu.OE_ACCEPT).mean(),
                            median_abs_ln_oe=g_['abs_ln_oe'].median()))
        cc = chance_ceiling(ev, 'p_lgbm', hs, R=min(R_SIM, 300))
        out.append(dict(hospital_set=setname, n=n, hospitals=len(hs), method='Chance ceiling (perfect calibration)',
                        pct_acceptable=cc['ceiling_pct_acceptable'], median_abs_ln_oe=cc['null_median_abs_ln_oe']))
lbt = pd.DataFrame(out)
tu.save_table(lbt.round(3), 'E_label_budget_all')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
d0 = lbt[lbt['hospital_set'].str.startswith('all')]
meths = ['Local intercept', 'Local logistic', 'EB (network prior, LOHO)', 'EB without global step (LOHO)',
         'No updating', 'Global logistic (LOHO)', 'Chance ceiling (perfect calibration)']
for ax, met, ttl in [(axes[0], 'pct_acceptable', '% hospitals with O/E in 0.8-1.2'),
                     (axes[1], 'median_abs_ln_oe', 'Median |ln O/E| (lower is better)')]:
    for i, k in enumerate(meths):
        g_ = d0[d0['method'] == k]
        ls = '-' if i < 4 else '--'
        ax.plot(g_['n'], g_[met], marker='o' if i < 4 else None, ms=4, ls=ls, color=(tu.SERIES + [tu.C['grey']] * 3)[i], label=k)
    ax.set_xscale('log'); ax.set_xticks(NS); ax.set_xticklabels([f'{n}\n(k={len([h for h in ELIG if nupd[h] >= n])})' for n in NS])
    ax.set_xlabel('Local patients used for updating (n); k = hospitals'); ax.set_title(ttl)
axes[0].legend(loc='lower right', fontsize=7)
tu.save_fig(fig, 'fig7_label_budget_rev'); plt.show()

## F. Meta-regression: proportion of between-hospital variance explained

In [ ]:
def meta_reg(ph, label):
    d = ph[ph['eligible']].merge(hinfo, on='hospital_id', how='left').dropna(subset=['ln_oe', 'ln_oe_se'])
    d['teaching'] = (d['teachingstatus'].astype(str).str.lower() == 'true').astype(float)
    X = pd.get_dummies(d[['numbedscategory', 'region']].fillna('Unknown'), drop_first=True, dtype=float)
    X['teaching'] = d['teaching']
    X['casemix_apache_x10'] = 10 * d['casemix_apache'].fillna(d['casemix_apache'].mean())
    X['mean_age_per10y'] = d['mean_age'] / 10
    X = sm.add_constant(X).astype(float)
    y_, v_ = d['ln_oe'].values, d['ln_oe_se'].values ** 2
    k, pp = X.shape
    W = np.diag(1 / v_); Xv = X.values
    XtWX_inv = np.linalg.inv(Xv.T @ W @ Xv)
    P = W - W @ Xv @ XtWX_inv @ Xv.T @ W
    QE = float(y_ @ P @ y_)
    tau2_res = max(0.0, (QE - (k - pp)) / np.trace(P))         # method of moments (DerSimonian-Laird type)
    tau2_tot = tu.meta_analysis(y_, np.sqrt(v_))['tau2']
    fit = sm.WLS(y_, X, weights=1 / (v_ + tau2_res)).fit(cov_type='fixed scale')
    res = pd.DataFrame(dict(model=label, term=fit.params.index, coef_ln_oe=fit.params.values, se=fit.bse.values,
                            p=fit.pvalues.values))
    info = dict(model=label, k=k, tau2_total=tau2_tot, tau2_residual=tau2_res,
                r2_tau2_explained=max(0.0, 1 - tau2_res / tau2_tot) if tau2_tot > 0 else np.nan, QE=QE, df=k - pp)
    return res, info
mr, mi = [], []
for lab_, col in [('LightGBM', 'p_lgbm'), ('APACHE IVa', 'apache_pred')]:
    d = pred if col == 'p_lgbm' else pred[pred['apache_pred'].notna()]
    r_, i_ = meta_reg(tu.per_hospital(d, col, full=False), lab_); mr.append(r_); mi.append(i_)
tu.save_table(pd.concat(mr).round(4), 'F_meta_regression')
tu.save_table(pd.DataFrame(mi).round(5), 'F_tau2_explained')

## G. Why is the MIMIC temporal-test AUROC higher than the development AUROC?

In [ ]:
pmx = pm.merge(m[['stay_id', 'first_careunit', 'age']], on='stay_id', how='left')
g_rows = []
for (st, ay), d in [((s_, 'all'), d_) for s_, d_ in pmx.groupby('set')] + list(pmx[pmx['set'] == 'temporal_test'].groupby(['set', 'anchor_year_group'])):
    lp = tu.logit(d['p_lgbm'].values)
    g_rows.append(dict(set=st, period=ay, n=len(d), mortality=d['mortality'].mean(), mean_pred=d['p_lgbm'].mean(),
                       sd_logit_pred=lp.std(), auroc=tu.roc_auc_score(d['mortality'], d['p_lgbm']),
                       mean_age=d['age'].mean()))
tu.save_table(pd.DataFrame(g_rows).round(4), 'G_temporal_auroc')
cm = pmx.groupby(['set', 'first_careunit']).agg(n=('mortality', 'size'), mortality=('mortality', 'mean')).reset_index()
cm['pct'] = 100 * cm['n'] / cm.groupby('set')['n'].transform('sum')
au = []
for (st, cu), d in pmx.groupby(['set', 'first_careunit']):
    if d['mortality'].sum() >= 20 and (1 - d['mortality']).sum() >= 20:
        au.append(dict(set=st, first_careunit=cu, auroc=tu.roc_auc_score(d['mortality'], d['p_lgbm'])))
cm = cm.merge(pd.DataFrame(au), on=['set', 'first_careunit'], how='left')
tu.save_table(cm.round(4), 'G_casemix_by_unit')
# Same final model scored on the development data with a 5-fold refit is what OOF does; OOF models see 80% of data.

## H. Second outcome: ICU stay of 7 days or more (same pipeline, same hyper-parameters)

In [ ]:
def raw(name):
    return pd.read_parquet(os.path.join(tu.DATA_RAW, name + '.parquet'))
mc = raw('mimic_cohort')[['stay_id', 'icu_los_h']].copy()
mc['stay_id'] = pd.to_numeric(mc['stay_id'], errors='coerce'); mc['icu_los_h'] = pd.to_numeric(mc['icu_los_h'], errors='coerce')
ec = raw('eicu_cohort')[['stay_id', 'unitdischargeoffset']].copy()
ec['stay_id'] = pd.to_numeric(ec['stay_id'], errors='coerce'); ec['unitdischargeoffset'] = pd.to_numeric(ec['unitdischargeoffset'], errors='coerce')
m2 = m.merge(mc.drop_duplicates('stay_id'), on='stay_id', how='left')
e2 = e.merge(ec.drop_duplicates('stay_id'), on='stay_id', how='left')
m2['prolonged'] = (m2['icu_los_h'] >= 168).astype(int)
e2['prolonged'] = (e2['unitdischargeoffset'] >= 7 * 1440).astype(int)
print('ICU stay >= 7 d: MIMIC', round(m2['prolonged'].mean(), 4), '| eICU', round(e2['prolonged'].mean(), 4),
      '| missing LOS', int(m2['icu_los_h'].isna().sum()), int(e2['unitdischargeoffset'].isna().sum()))
dev2 = m2[m2['period'] == 'dev']; tt2 = m2[m2['period'] != 'dev']
def fit_los():
    return lgb.LGBMClassifier(n_estimators=bp['n_estimators'], subsample_freq=1, random_state=tu.SEED, verbose=-1,
                              n_jobs=-1, **bp['params']).fit(dev2[FEATS].astype(float), dev2['prolonged'].astype(int))
mdl = tu.cached('los_model', fit_los)
e2['p_los'] = mdl.predict_proba(e2[FEATS].astype(float))[:, 1]
p_tt = mdl.predict_proba(tt2[FEATS].astype(float))[:, 1]
h_rows = []
for lab_, y_, p_, cl in [('MIMIC temporal test', tt2['prolonged'].values, p_tt, None),
                         ('eICU all (zero adaptation)', e2['prolonged'].values, e2['p_los'].values, e2['hospital_id'].values)]:
    ci, _ = tu.bootstrap(y_, p_, clusters=cl, B=tu.n_boot(500), metrics=['auroc', 'oe', 'citl', 'slope', 'ici'])
    for _, r in ci.iterrows():
        h_rows.append(dict(setting=lab_, prevalence=y_.mean(), metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi']))
tu.save_table(pd.DataFrame(h_rows).round(4), 'H_los_transport')
pl = e2[['stay_id', 'hospital_id', 'year', 'prolonged', 'p_los']].merge(pred[['stay_id', 'split']], on='stay_id')
ph2 = tu.per_hospital(pl, 'p_los', ycol='prolonged')
het = tu.summarise_hospitals(ph2, 'LightGBM, ICU stay >= 7 d')
H2 = ph2.loc[ph2['eligible'], 'hospital_id'].tolist()
c2 = chance_ceiling(pl.rename(columns={'prolonged': '_y'}), 'p_los', H2)
v2 = ph2[ph2['expected'] >= 1]
het['ceiling_pct_acceptable'] = c2['ceiling_pct_acceptable']
het['pct_outside_95'] = 100 * tu.outside_funnel(v2, 0.95).mean()
het['pct_outside_998'] = 100 * tu.outside_funnel(v2, 0.998).mean()
het['null_pct_outside_95'] = c2['null_pct_outside_95']
tu.save_table(het.round(4), 'H_los_heterogeneity')
o2, evs2, el2, info2 = strategy_summary(pl[pl['split'] == 'update'].reset_index(drop=True),
                                        pl[pl['split'] == 'eval'].reset_index(drop=True), 'p_los',
                                        'ICU stay >= 7 d, temporal 2014->2015', ycol='prolonged')
s2 = pd.DataFrame(o2)
a2 = pd.DataFrame({s: tu.per_hospital(evs2[evs2['hospital_id'].isin(el2)], s, ycol='prolonged', min_n=0, min_events=0,
                                      full=False).set_index('hospital_id')['ln_oe'] for s in KEY}).abs()
s2['p_wilcoxon_vs_eb'] = [np.nan if s == 'S3_eb' else wilcoxon(a2['S3_eb'], a2[s]).pvalue for s in KEY]
tu.save_table(s2.round(4), 'H_los_strategies')
print(info2)

In [ ]:
tu.finish_notebook({
    'chance_ceiling': ceil.round(4).to_dict('records'),
    'random_split': rs.round(4).to_dict('records'),
    'strategies_extended': summ.drop(columns=['key']).round(4).to_dict('records'),
    'wilcoxon_holm': wx.round(5).to_dict('records'),
    'glmm_all': dict(alpha=gl.alpha, beta=gl.beta, tau=gl.tau, converged=gl.ok),
    'leave_region_out': lro_tab.round(4).to_dict('records'),
    'tau2_explained': pd.DataFrame(mi).round(5).to_dict('records'),
    'temporal_auroc': pd.DataFrame(g_rows).round(4).to_dict('records'),
    'los_heterogeneity': het.round(4).to_dict('records'),
    'los_strategies': s2.round(4).to_dict('records'),
})